# Deepfake & Face-Swap Detection Model Training

This notebook provides a complete pipeline to train, validate, evaluate, and export deep learning models for facial forgery and face-swap detection.

### Pipeline Steps:
1. **Environment & Device Configuration**
2. **Dataset Setup & Forensic Data Augmentation** (compression simulation, blur, jitter)
3. **Model Selection**: MesoNet-4 vs. MesoInception-4 vs. MobileNetV3
4. **Training Loop with Early Stopping & LR Scheduling**
5. **Evaluation Metrics**: ROC-AUC, Precision, Recall, F1-Score, and Confusion Matrix
6. **Model Checkpoint Export** (`.pth` weights for real-time deployment in `app.py`)

In [ ]:
import os
import sys
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, roc_curve

# Ensure project root is in sys.path
project_root = Path(os.getcwd()).resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.deepfake_detector import Meso4, MesoInception4, MobileNetDeepfake

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Computation Device: {device}')

## 1. Forensic Data Augmentation & Preprocessing
Deepfake models easily overfit to generator-specific noise patterns. Applying realistic compression, motion blur, and color jitter forces the network to learn structural facial inconsistencies rather than superficial compression artifacts.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 2. Dataset Loader
Loads authentic (`real/`) and manipulated (`fake/`) face images from `data/processed/` or custom folders.

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        
        real_dir = self.root_dir / 'real'
        fake_dir = self.root_dir / 'fake'
        
        if real_dir.exists():
            for p in real_dir.glob('*.*'):
                if p.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    self.samples.append((str(p), 0.0))
        if fake_dir.exists():
            for p in fake_dir.glob('*.*'):
                if p.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    self.samples.append((str(p), 1.0))
                    
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor([label], dtype=torch.float32)

print('Dataset class loaded.')

## 3. Model Architecture & Optimizer Setup
Choose between **Meso4** (optimal for lightweight real-time CPU deployment) or **MobileNetDeepfake**.

In [ ]:
model = Meso4(num_classes=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Initialized Meso4 model with {total_params:,} trainable parameters.')

## 4. Training & Validation Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    return total_loss / max(1, total), correct / max(1, total)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(probs.flatten())
            all_labels.extend(labels.cpu().numpy().flatten())
            
    return total_loss / max(1, len(all_labels)), np.array(all_preds), np.array(all_labels)

print('Training and evaluation functions ready.')

## 5. Metrics & ROC-AUC Evaluation

In [ ]:
def print_metrics(preds, targets):
    binary_preds = (preds >= 0.5).astype(int)
    acc = accuracy_score(targets, binary_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(targets, binary_preds, average='binary', zero_division=0)
    auc = roc_auc_score(targets, preds) if len(np.unique(targets)) > 1 else 0.5
    
    print('----------------------------------------')
    print(f'Accuracy:  {acc * 100:.2f}%')
    print(f'Precision: {prec * 100:.2f}%')
    print(f'Recall:    {rec * 100:.2f}%')
    print(f'F1-Score:  {f1:.4f}')
    print(f'ROC-AUC:   {auc:.4f}')
    print('----------------------------------------')
    return acc, auc

print('Metrics evaluator ready.')

## 6. Exporting Model Checkpoints
Save the trained model weights directly to the `models/` directory for use in `app.py`.

In [ ]:
save_path = project_root / 'models' / 'meso4_trained.pth'
save_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), str(save_path))
print(f'Trained model weights saved to: {save_path}')
print('To run real-time inference with this checkpoint:')
print(f'  python app.py --weights {save_path}')